In [ ]:
# Standard libraries
import numpy as np
from pathlib import Path
import plotly.express as px
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from functools import partial
import pandas as pd

# Specific libraries
import nibabel as nib
import seaborn as sns
import colorcet

# custom functions
import file_io as fio
import registration as reg
import temperature as temp
import calib_utils as cal

In [ ]:
parent_dir = Path("./calibration")
derive_dir = parent_dir / "derivatives"
testname = parent_dir.stem
plot_dir = parent_dir / "plots"
plot_dir.mkdir(exist_ok=True)

calib_dir = parent_dir / "calib"
calib_dir.mkdir(exist_ok=True)

param_fname = derive_dir / (testname + "_" + "params.h5")
scan_fname = derive_dir / (testname + "_" + "RAW_MPPCA_REG.nii.gz")
mask_fname = derive_dir / (testname + "_" + "MASK.nii.gz")

# Load in sequence parameters
df = pd.read_hdf(param_fname, key="df", mode="r")

# Load in scan data and add into data frame
scan_nii = nib.load(scan_fname)
scan_data = scan_nii.get_fdata()
scan_aff = scan_nii.affine
scan_spacing = reg.spacing_from_affine(scan_aff)

scan_data_reshuffled = np.transpose(scan_data,[3,0,1,2])
df["image"] = list(scan_data_reshuffled)
df["adc"] = df["temperature"].apply(partial(temp.water_diffusivity, polynomial=True))

# Load in mask
mask_nii = nib.load(mask_fname)
mask = mask_nii.get_fdata().astype(int)

# Create a mask slightly inset from threshold based mask to avoid boundary artefacts.
shrink_dist = 6 # mm
core_mask = cal.shrink_mask_mm(mask, voxel_spacing=scan_spacing, margin_mm=shrink_dist)

In [ ]:
df.sort_values("scan_time", inplace=True, ignore_index=True)

# Put the b=0 image on b=0 rows, NaN elsewhere
b0_threshold = 5
df["s0_image"] = df["image"].where(df["b_value"] < b0_threshold)

# Forward-fill so each DWI scan gets the most recent preceding b=0 image
df["s0_image"] = df["s0_image"].ffill()

# Keep only diffusion-weighted scans
dwi_df = df[df["b_value"] > b0_threshold].copy()

# Optional safety check: drop rows that do not have a preceding b=0 scan
dwi_df = dwi_df[dwi_df["s0_image"].notna()].copy()

dwi_df["bmap"] = dwi_df.apply(lambda row: cal.bmap_3d(s0=row["s0_image"], 
                                                      s=row["image"], 
                                                      adc=row["adc"], 
                                                      mask=core_mask), 
                                                      axis=1)
dwi_df["cmap"] = dwi_df["bmap"]/dwi_df["b_value"]

# Estimate gradient maps from b-value maps assuming Stejskal-Tanner Equation applies.
dwi_df["gmap2"] = dwi_df.apply(lambda row: cal.b2grad2(b=row["bmap"],
                                              delta_small=row["dwi_grad_duration"],
                                              delta_big=(row["TE"] / 2),
                                              mask=core_mask),
                                              axis=1)

# Estimate a smoothed version of each gradient map for fitting
dwi_df["gmap2_bspline"] = dwi_df.apply(lambda row: cal.bspline_estimation(data=row["gmap2"], spacing=row["spacing"], mask=core_mask), axis=1)

In [ ]:
dur = 0.04
subset_df = dwi_df[dwi_df["dwi_grad_duration"]==dur].copy()

input_gvec = np.stack(subset_df["grad_vec"].values, axis=0).astype(np.float32)
input_gmaps = np.stack(subset_df["gmap2_bspline"].values, axis=-1).astype(np.float32)

A, sigma = cal.calc_calib_maps(input_gvec, input_gmaps, core_mask, as_image=False)
A_nii = cal.A_as_nifti(A, mask=core_mask, affine=scan_aff, sigma=sigma, P=6)
A_img = A_nii.get_fdata()
fio.save_nifti(A_nii, calib_dir, testname + "_" + f"A_DUR_{int(dur*1e3):.0f}.nii.gz")

In [ ]:
# -------------------------
# Split data
# -------------------------
A_q = A_img[..., 3:]
A_l = A_img[..., :3]

# Deviations
A_q_avg = np.mean(A_q[core_mask == 1], axis=0)
A_l_avg = np.mean(A_l[core_mask == 1], axis=0)
print(A_q_avg)

A_q_dev = np.zeros_like(A_q)
A_l_dev = np.zeros_like(A_l)

A_q_dev[core_mask == 1] = A_q[core_mask == 1] - A_q_avg

# -------------------------
# Layout
# -------------------------
fig = plt.figure(figsize=(16, 7))
gs = fig.add_gridspec(
    2, 5,
    width_ratios=[2.5, 1, 1, 1, 0.1],
    wspace=0.25,
    hspace=0.35
)

ax_kde = [fig.add_subplot(gs[i, 0]) for i in range(2)]
ax_im = [[fig.add_subplot(gs[i, j]) for j in range(1, 4)] for i in range(2)]
cax = fig.add_subplot(gs[:, 4])

# -------------------------
# Settings
# -------------------------
kde_lim = 2e-5
i_slice = A_img.shape[2] // 2
kde_colors = plt.get_cmap("Blues")(np.linspace(0.5, 0.9, 3))

# -------------------------
# KDE plots
# -------------------------
for i in range(3):
    sns.kdeplot(
        A_q_dev[..., i][core_mask == 1],
        clip=(-kde_lim, kde_lim),
        color=kde_colors[i],
        ax=ax_kde[0],
        lw=0.8,
        label="XYZ"[i]
    )

    sns.kdeplot(
        A_l[..., i][core_mask == 1],
        clip=(-kde_lim, kde_lim),
        color=kde_colors[i],
        ax=ax_kde[1],
        lw=0.8,
        label="XYZ"[i]
    )

for i, ax in enumerate(ax_kde):
    ax.set_ylabel("Density", fontstyle="italic", fontsize=10, rotation=0)
    ax.yaxis.set_label_coords(-0.08, 1.)
    ax.set_ylim(0, 80000)
    ax.set_yticks(np.linspace(0, 70000, num=8, endpoint=True))
    ax.set_xlim(-kde_lim, kde_lim)
    ax.grid(alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize=10)
    ax.legend(frameon=False, fontsize=10)

ax_kde[0].set_xlabel(r"$C_{Gi} - \mu_{Gi}$", fontsize=12)
ax_kde[1].set_xlabel(r"$C_{Xi}$", fontsize=12)

# -------------------------
# Shared image scaling
# -------------------------
lim = np.max([
    np.abs(A_q_dev[core_mask == 1]).max(),
    np.abs(A_l_dev[core_mask == 1]).max()
]) * 1e3

# -------------------------
# Image plotting helper
# -------------------------
def plot_row(ax_list, data):
    ims = []
    for i, ax in enumerate(ax_list):

        sl = data[:, :, i_slice, i] * 1e3
        sl = np.rot90(sl, k=1)[:, 7:-8]

        mask = np.rot90(core_mask[:, :, i_slice], k=1)[:, 7:-8]
        overlay = np.full_like(sl, np.nan)
        overlay[mask == 0] = 1

        im = ax.imshow(sl, cmap=colorcet.cm.coolwarm, vmin=-lim, vmax=lim)
        ax.imshow(overlay, cmap="Greys_r", vmin=0, vmax=1)

        ax.set_axis_off()
        ims.append(im)

    return ims[-1]

# -------------------------
# Plot rows
# -------------------------
im_top = plot_row(ax_im[0], A_q_dev)
im_bot = plot_row(ax_im[1], A_l)

# -------------------------
# Colorbar
# -------------------------
cbar = fig.colorbar(im_top, cax=cax)
cbar.ax.set_title("$\\times 10^{-3}$", fontsize=12, pad=10)
cbar.ax.tick_params(labelsize=12)

# -------------------------
# Row labels
# -------------------------
row_labels = [r"$C_G$", r"$C_X$"]

for i in range(2):
    # use full row span (all axes in that row)
    row_axes = [ax_kde[i]] + ax_im[i]

    y0 = min(ax.get_position().y0 for ax in row_axes)
    y1 = max(ax.get_position().y1 for ax in row_axes)
    y_center = (y0 + y1) / 2

    # place safely left of ALL axes (not just KDE)
    x_left = min(ax.get_position().x0 for ax in row_axes)

    fig.text(
        x_left - 0.06,   # <- key spacing control
        y_center,
        row_labels[i],
        rotation=0,
        va="center",
        ha="center",
        fontsize=20
    )

# -------------------------
# Column headers
# -------------------------
col_labels = ["X", "Y", "Z"]
for i, ax in enumerate(ax_im[0]):
    pos = ax.get_position()
    fig.text(
        pos.x0 + pos.width / 2,
        pos.y1 + 0.02,
        col_labels[i],
        ha="center",
        va="bottom",
        fontsize=20
    )

fmt="png"
plot_fname = plot_dir / (testname + "_" + f"CMAPS_DUR_{int(dur*1e3):.0f}.{fmt}")
fig.savefig(fname=plot_fname, dpi=600, format=fmt, bbox_inches='tight', pad_inches=0.01)

In [ ]:
# -----------------------------------------------------------------------------
# Prepare repeat index for leave-one-out over repeats
# -----------------------------------------------------------------------------
dur = 0.04
loo_df = dwi_df[dwi_df["dwi_grad_duration"]==dur].copy()
loo_df["repeat_idx"] = loo_df.groupby("seq_name").cumcount()

# Number of repeats available per sequence
repeat_counts = loo_df.groupby("seq_name").size()
n_repeats = repeat_counts.min()

if repeat_counts.nunique() != 1:
    print("Warning: not all seq_name groups have the same number of repeats.")
    print("Using n_repeats =", n_repeats, "(minimum count across seq_name groups)")

# Restrict to complete repeats only if some sequences have fewer repeats
loo_df = loo_df[loo_df["repeat_idx"] < n_repeats].copy()

# Consistent sequence ordering
seq_order = sorted(loo_df["seq_name"].unique())
dur_order = [float(seq[-2:]) for seq in seq_order]
dw_axis_order = [seq[5].upper() for seq in seq_order]
bval_order = [int(seq[7:10]) for seq in seq_order]
n_seq = len(seq_order)

# Storage across folds
measured_folds = []
predicted_folds = []
corrected_folds = []
bval_folds = []

# -----------------------------------------------------------------------------
# Leave-one-out across repeat index
# -----------------------------------------------------------------------------
for loo_idx in range(n_repeats):
    print(f"LOO fold {loo_idx+1}/{n_repeats}")

    test_df = loo_df[loo_df["repeat_idx"] == loo_idx].copy()
    train_df = loo_df[loo_df["repeat_idx"] != loo_idx].copy()

    # Ensure consistent ordering
    test_df["seq_name"] = pd.Categorical(test_df["seq_name"], categories=seq_order, ordered=True)
    train_df["seq_name"] = pd.Categorical(train_df["seq_name"], categories=seq_order, ordered=True)

    test_df = test_df.sort_values("seq_name").reset_index(drop=True)
    train_df = train_df.sort_values("seq_name").reset_index(drop=True)

    # Train calibration model
    train_gvecs = np.stack(train_df["grad_vec"].values, axis=0).astype(np.float32)
    train_maps = np.stack(train_df["gmap2_bspline"].values, axis=-1).astype(np.float32)

    A, sigma = cal.calc_calib_maps(train_gvecs, train_maps, core_mask, as_image=False)

    # Test on held-out repeat
    test_gvecs = np.stack(test_df["grad_vec"].values, axis=0).astype(np.float32)
    delta_small = test_df["dwi_grad_duration"].to_numpy(dtype=np.float32).reshape(1, 1, 1, -1)
    delta_big = ((test_df["TE"] / 2)).to_numpy(dtype=np.float32).reshape(1, 1, 1, -1)
    eval_cmaps = np.stack(test_df["cmap"].values, axis=-1).astype(np.float32)
    eval_bvalues = test_df["b_value"].to_numpy(dtype=np.float32).reshape(1,1,1,-1)

    test_X = np.apply_along_axis(cal.x_from_gvec, arr=test_gvecs, axis=1)
    test_Z = np.apply_along_axis(partial(cal.z_from_x, sigma=sigma), arr=test_X, axis=1)

    pred_g2maps = np.zeros_like(eval_cmaps, dtype=np.float32)
    pred_g2maps[core_mask == 1] = (test_Z @ A).T

    pred_bmaps = cal.grad22b(pred_g2maps, delta_small=delta_small, delta_big=delta_big, mask=core_mask)
    pred_cmaps = pred_bmaps / eval_bvalues

    corr_cmaps = np.zeros_like(eval_cmaps, dtype=np.float32)
    valid_mask = (eval_cmaps > 0) & (pred_cmaps > 0) & (core_mask[..., None] == 1)
    corr_cmaps[valid_mask] = eval_cmaps[valid_mask] / pred_cmaps[valid_mask]
    
    measured_folds.append(eval_cmaps)
    predicted_folds.append(pred_cmaps)
    corrected_folds.append(corr_cmaps)
    bval_folds.append(eval_bvalues.squeeze())

# -----------------------------------------------------------------------------
# Stack fold results
# Shape will be:
#   measured_all  -> (X, Y, Z, n_seq, n_repeats)
# -----------------------------------------------------------------------------
measured_all = np.stack(measured_folds, axis=-1)
predicted_all = np.stack(predicted_folds, axis=-1)
corrected_all = np.stack(corrected_folds, axis=-1)
bvals_all = np.stack(bval_folds, axis=-1)

# Average across LOO folds for imshow
mean_measured = np.mean(measured_all, axis=-1)
mean_predicted = np.mean(predicted_all, axis=-1)
mean_corrected = np.mean(corrected_all, axis=-1)


In [ ]:
i_slice = mean_measured.shape[2] // 2
lim = 0.4

# -------------------------
# Layout assumptions
# -------------------------
n_figures = 3          # rows (X, Y, Z)
seq_per_figure = 4    # columns

# -------------------------
# Consistent font settings
# -------------------------
fs_seq_title = 20
fs_im_title = 12
fs_ticks = 12
fs_legend = 9
fs_cbar = 14
kde_lw = 0.8

for dur in np.unique(dur_order):

    i_dur = np.where(np.array(dur_order) == dur)[0]

    dur_seq_order = [seq_order[i] for i in i_dur]
    dur_dw_axis_order = [dw_axis_order[i] for i in i_dur]
    dur_bval_order = [bval_order[i] for i in i_dur]

    dur_measured = mean_measured[..., i_dur]
    dur_predicted = mean_predicted[..., i_dur]
    dur_corrected = mean_corrected[..., i_dur]

    dur_measured_all = measured_all[..., i_dur, :]
    dur_corrected_all = corrected_all[..., i_dur, :]

    # -------------------------
    # Create ONE figure per duration
    # -------------------------
    fig = plt.figure(figsize=(25, 4.5 * n_figures))

    outer_gs = fig.add_gridspec(
        n_figures, seq_per_figure,
        wspace=0.25,
        hspace=0.35
    )

    # -------------------------
    # Loop over rows (axes)
    # -------------------------
    for fig_idx in range(n_figures):

        seq_start = fig_idx * seq_per_figure
        seq_end = seq_start + seq_per_figure

        image_axes_row = []
        last_im = None

        # -------------------------
        # Loop over columns (b-values)
        # -------------------------
        for j, i_seq in enumerate(range(seq_start, seq_end)):

            inner_gs = outer_gs[fig_idx, j].subgridspec(
                2, 3,
                hspace=0.15,
                wspace=0.05
            )

            ax1 = fig.add_subplot(inner_gs[0, 0])
            ax2 = fig.add_subplot(inner_gs[0, 1])
            ax3 = fig.add_subplot(inner_gs[0, 2])
            ax4 = fig.add_subplot(inner_gs[1, :])

            meas = np.rot90(dur_measured[:, :, i_slice, i_seq], k=1)[:, :-1]
            pred = np.rot90(dur_predicted[:, :, i_slice, i_seq], k=1)[:, :-1]
            corr = np.rot90(dur_corrected[:, :, i_slice, i_seq], k=1)[:, :-1]

            mask_slice = np.rot90(core_mask[:, :, i_slice], k=1)[:, :-1]
            plot_mask = np.full_like(meas, np.nan, dtype=float)
            plot_mask[mask_slice == 0] = 1

            im_axes = [ax1, ax2, ax3]
            im_data = [meas, pred, corr]
            im_labels = ["Measured", "Predicted", "Corrected"]

            for ax, data, lab in zip(im_axes, im_data, im_labels):
                im = ax.imshow(
                    data,
                    vmin=1 - lim,
                    vmax=1 + lim,
                    cmap=colorcet.cm.coolwarm,
                    interpolation=None
                )

                ax.imshow(
                    plot_mask,
                    vmin=0,
                    vmax=1,
                    cmap="Greys_r",
                    interpolation="nearest"
                )

                ax.set_axis_off()

                ax.text(
                    0.5, -0.0, lab,
                    transform=ax.transAxes,
                    ha="center",
                    va="top",
                    fontsize=fs_im_title
                )

                image_axes_row.append(ax)
                last_im = im

            # -------------------------
            # KDE plots
            # -------------------------
            meas_all = dur_measured_all[:, :, :, i_seq, :]
            corr_all = dur_corrected_all[:, :, :, i_seq, :]

            for i in range(meas_all.shape[-1]):
                sns.kdeplot(
                    meas_all[..., i][core_mask == 1],
                    color="darkorange",
                    clip=(1 - lim, 1 + lim),
                    ax=ax4,
                    lw=kde_lw
                )

                sns.kdeplot(
                    corr_all[..., i][core_mask == 1],
                    color="cornflowerblue",
                    clip=(1 - lim, 1 + lim),
                    ax=ax4,
                    lw=kde_lw
                )
                meas_patch = mpatches.Patch(facecolor='darkorange', edgecolor="k", linewidth=0.5, label="Measured")
                corr_patch = mpatches.Patch(facecolor='cornflowerblue', edgecolor="k", linewidth=0.5, label="Corrected")

            ax4.set_xlim((1 - lim, 1 + lim))
            ax4.set_ylim((0, 35))
            ax4.grid(alpha=0.25)
            ax4.tick_params(labelsize=fs_ticks)
            ax4.legend(handles=[meas_patch, corr_patch], fontsize=fs_legend, frameon=False, loc="upper left")
            ax4.set_xlabel(r"$c(\mathbf{r})$", fontsize=fs_im_title)
            ax4.spines["top"].set_visible(False)
            ax4.spines["right"].set_visible(False)
            

            # -------------------------
            # Column titles (top row only)
            # -------------------------
            if fig_idx == 0:
                ax2.set_title(
                    rf"$b = {dur_bval_order[i_seq]}$",
                    fontsize=fs_seq_title,
                    pad=10
                )

        # -------------------------
        # Row label (axis)
        # -------------------------
        row_label = "XYZ"[fig_idx]

        y_positions = [
            ax.get_position().y0
            for ax in image_axes_row
        ]
        y_center = np.mean(y_positions)

        left_edge = min(ax.get_position().x0 for ax in image_axes_row)

        fig.text(
            left_edge - 0.03,
            y_center,
            row_label,
            va="center",
            ha="right",
            fontsize=fs_seq_title,
            fontstyle="italic",
            rotation=0
        )

        # -------------------------
        # Colorbar per row
        # -------------------------
        x1 = max(ax.get_position().x1 for ax in image_axes_row)
        y0 = min(ax.get_position().y0 for ax in image_axes_row)
        y1 = max(ax.get_position().y1 for ax in image_axes_row)

        cax = fig.add_axes([x1 + 0.01, y0, 0.004, y1 - y0])
        cbar = fig.colorbar(last_im, cax=cax)
        cbar.ax.set_title(r"$c(r)$", fontsize=fs_cbar, pad=10)
        cbar.ax.tick_params(labelsize=fs_ticks)

    col_edges = []
    row_edges = []
    offset_y = 0.025
    offset_x = 0.005
    pad = 0.01

    # column boundaries (between b-value blocks)
    for j in range(seq_per_figure - 1):
        left = outer_gs[0, j].get_position(fig)
        right = outer_gs[0, j + 1].get_position(fig)
        x = (left.x1 + right.x0) / 2 - offset_x
        col_edges.append(x)

    # row boundaries (between X/Y/Z)
    for i in range(n_figures - 1):
        top = outer_gs[i, 0].get_position(fig)
        bottom = outer_gs[i + 1, 0].get_position(fig)
        y = (bottom.y1 + top.y0) / 2 - offset_y
        row_edges.append(y)

    for x in col_edges:
        for i in range(n_figures):

            top_cell = outer_gs[i, 0].get_position(fig)
            bottom_cell = outer_gs[i, 0].get_position(fig)

            y_top = top_cell.y1 - offset_y + pad
            y_bottom = bottom_cell.y0 - offset_y - pad

            fig.add_artist(Line2D(
                [x, x],
                [y_bottom, y_top],
                transform=fig.transFigure,
                linestyle="--",
                linewidth=1.2,
                color="black",
                alpha=0.5
            ))

    for y in row_edges:
        for j in range(seq_per_figure):

            left_cell = outer_gs[0, j].get_position(fig)
            right_cell = outer_gs[0, j].get_position(fig)

            x_left = left_cell.x0 - offset_x - pad
            x_right = right_cell.x1 - offset_x + pad

            fig.add_artist(Line2D(
                [x_left, x_right],
                [y, y],
                transform=fig.transFigure,
                linestyle="--",
                linewidth=1.2,
                color="black",
                alpha=0.35
            ))
    # -------------------------
    # Single suptitle
    # -------------------------
    plt.suptitle(rf"$\delta = {int(dur):.0f}\ \mathrm{{ms}}$", 
                fontsize=fs_seq_title,
                y=0.95)

    # -------------------------
    # Save
    # -------------------------
    fmt = "png"
    plot_fname = plot_dir / (
        testname + "_" + f"DURSPECIFIC_loo_dur_{int(dur):.0f}.{fmt}"
    )

    plt.show()
    fig.savefig(
        fname=plot_fname,
        dpi=600,
        format=fmt,
        bbox_inches="tight",
        pad_inches=0.025
    )
